# GuitarMidi-LV2 Library
 # Copyright (C) 2026 Gerald Mwangi
 #
 # This program is free software; you can redistribute it and/or
 # modify it under the terms of the GNU Lesser General Public
 # License as published by the Free Software Foundation; either
 # version 2 of the License, or (at your option) any later version.
 #
 # This program is distributed in the hope that it will be useful,
 # but WITHOUT ANY WARRANTY; without even the implied warranty of
 # MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the GNU
 # Lesser General Public License for more details.
 #
 # You should have received a copy of the GNU Lesser General
 # Public License along with this program; if not, write to the
 # Free Software Foundation, Inc., 51 Franklin Street, Fifth Floor,
 # Boston, MA  02110-1301  USA

In [2]:
import tensorflow as tf
import os
import glob,re
import random
import numpy as np
from model import build_1d_cnn_model
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES, fast_gpu_map,parse_filtered_audio_record
cnn_model=build_1d_cnn_model(1,INPUT_SHAPE,37,False)
cnn_model.summary()
#tf.keras.utils.plot_model(cnn_model,to_file='cnn_model.png',show_shapes=True)
cnn_model.load_weights('/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/checkpoints/guitarmidi_staticmapping_epoch168_valAcc0.9756_valPrec0.8755_valRecall0.7175.keras')#
#cnn_model.load_weights('guitarmidi.keras')


input_filepaths = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/training_subset/training_subset_electric'#sorted(glob.glob(os.path.join(input_data_dir, '**', 'input', 'data.tfrecord'), recursive=True))
input_filepaths=glob.glob(os.path.join(input_filepaths, '**', '*.tfrecord'), recursive=True)
input_filepaths = sorted(input_filepaths,key=lambda file: int(re.findall('\\d+',os.path.basename(file))[0]) )
random.shuffle(input_filepaths)
# train_dataset = tf.data.Dataset.from_tensor_slices((input_filepaths))
# train_dataset=train_dataset.shuffle(buffer_size=len(input_filepaths))
# train_dataset=train_dataset.take(100)
# train_dataset = train_dataset.map(tf_load_sample_from_files, num_parallel_calls=tf.data.AUTOTUNE)
def representative_data_gen():
    # Use TFRecordDataset to actually read the files
    # We only need a few samples to calibrate quantization
    raw_dataset = tf.data.TFRecordDataset(input_filepaths).map(parse_filtered_audio_record).shuffle(buffer_size=len(input_filepaths)).take(1000)
    
    # Map using your existing loading function
    calib_dataset = raw_dataset.map(lambda it,ot: fast_gpu_map(it,ot, training=False))
    
    for input_value, _ in calib_dataset.batch(1):
        # input_value is the (312, 256, 1) tensor
        yield [input_value]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
#converter.representative_dataset = representative_data_gen
tflite_model = converter.convert()


# converter=tf.lite.TFLiteConverter.from_keras_model(cnn_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model=converter.convert()

with open('guitarmidi.tflite','wb') as f:
    f.write(tflite_model)
print("TFLite model saved as guitarmidi.tflite")

Image height:  148
Before string split: (1, 148, 512), max_x=148.0
String slice ranges (in time steps):  [(0, 52), (20, 72), (40, 92), (60, 112), (76, 128), (96, 148)]
String 0: slicing from 0 to 52 (max_x=148.0)
String 1: slicing from 20 to 72 (max_x=148.0)
String 2: slicing from 40 to 92 (max_x=148.0)
String 3: slicing from 60 to 112 (max_x=148.0)
String 4: slicing from 76 to 128 (max_x=148.0)
String 5: slicing from 96 to 148 (max_x=148.0)
After string split:  [(1, 13, 64), (1, 13, 64), (1, 13, 64), (1, 13, 64), (1, 13, 64), (1, 13, 64)]


Model: "guitar_note_detector"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_spectrogram   │ (1, 148, 256, 1)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_mean          │ (1, 148, 256, 1)  │          0 │ input_spectrogra… │
│ (AveragePooling2D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_contrast      │ (1, 148, 256, 1)  │          0 │ input_spectrogra… │
│ (Subtract)          │                   │            │ local_mean[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_to_2d       │ (1, 148, 256, 1)  │          0 │ local_contrast[0… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_conv… │ (1, 148, 64, 8)   │        136 │ reshape_to_2d[0]… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_bn    │ (1, 148, 64, 8)   │         32 │ freq_compress_co… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_act   │ (1, 148, 64, 8)   │          0 │ freq_compress_bn… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_drop  │ (1, 148, 64, 8)   │          0 │ freq_compress_ac… │
│ (SpatialDropout2D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_to_1d       │ (1, 148, 512)     │          0 │ freq_compress_dr… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ harm_pos_enc        │ (1, 148, 512)     │     20,993 │ reshape_to_1d[0]… │
│ (HarmonicPositiona… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ln1      │ (1, 148, 512)     │      1,024 │ harm_pos_enc[0][… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_mha      │ (1, 148, 512)     │    131,776 │ tfm_block1_ln1[0… │
│ (MultiHeadAttentio… │                   │            │ tfm_block1_ln1[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_attn_add │ (1, 148, 512)     │          0 │ harm_pos_enc[0][… │
│ (Add)               │                   │            │ tfm_block1_mha[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ln2      │ (1, 148, 512)     │      1,024 │ tfm_block1_attn_… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ffn1     │ (1, 148, 128)     │     65,664 │ tfm_block1_ln2[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ffn_drop │ (1, 148, 128)     │          0 │ tfm_block1_ffn1[… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ffn2     │ (1, 148, 512)     │     66,048 │ tfm_block1_ffn_d

 Total params: 813,743 (3.10 MB)

 Trainable params: 810,271 (3.09 MB)

 Non-trainable params: 3,472 (13.56 KB)

INFO:tensorflow:Assets written to: /tmp/tmp8bwvbbwx/assets


INFO:tensorflow:Assets written to: /tmp/tmp8bwvbbwx/assets


Saved artifact at '/tmp/tmp8bwvbbwx'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 148, 256, 1), dtype=tf.float32, name='input_spectrogram')
Output Type:
  TensorSpec(shape=(1, 37), dtype=tf.float32, name=None)
Captures:
  138077027579856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138077027578128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138077027579088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138077027577936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138077027577552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138077027578896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138077027581200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138077027581008: TensorSpec(shape=(148,), dtype=tf.int32, name=None)
  138077027578704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138077027579664: TensorSpec(shape=(148,), dtype=tf.int32, name=None)
  138077027577

W0000 00:00:1786790108.800916 1110560 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1786790108.800937 1110560 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1786790108.801156 1110560 reader.cc:83] Reading SavedModel from: /tmp/tmp8bwvbbwx
I0000 00:00:1786790108.805592 1110560 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1786790108.805610 1110560 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp8bwvbbwx
I0000 00:00:1786790108.868575 1110560 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1786790109.296631 1110560 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmp8bwvbbwx
I0000 00:00:1786790109.407495 1110560 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 606350 microseconds.
I0000 00:00:1786790109.944282 1110560 flatbuffer_export.cc:4302] Estimated count of arithmetic ops: 136.229 M  ops, equivalently 68.114 M  MACs
